# NOAA NCEI County Climate Raw Data Quality

Monthly county temperature and precipitation observations.

The audit queries the current `raw.*` ingestion tables directly; transformed tables are out of scope.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "data" / "housing_predict.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
con = duckdb.connect(str(ROOT / "data" / "housing_predict.duckdb"), read_only=True)
TABLES = ["climate"]
KEYS = {"climate":["fips","parameter","year_month"]}
YEAR_COLUMNS = {"climate":"year"}
EXPECTED_YEARS = set(range(2010, 2025))
def qi(x): return '"' + x.replace('"', '""') + '"'
def columns(t):
    return con.execute("SELECT column_name FROM information_schema.columns WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [t]).df()["column_name"].tolist()
available = set(con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='raw'").df()["table_name"])
assert set(TABLES) <= available, f"Missing raw tables: {set(TABLES)-available}"


## Inventory, coverage, keys, and sampled missingness

In [2]:
inventory, key_checks, coverage, missing = [], [], [], []
for table in TABLES:
    cols = columns(table)
    n = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    inventory.append({"table": table, "rows": n, "columns": len(cols)})
    key = KEYS[table]
    key_sql = ", ".join(qi(c) for c in key)
    invalid = " OR ".join(f"{qi(c)} IS NULL OR trim(cast({qi(c)} AS VARCHAR))=''" for c in key)
    key_checks.append({
        "table": table, "candidate_key": " + ".join(key),
        "invalid_key_rows": con.execute(f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}").fetchone()[0],
        "duplicate_key_groups": con.execute(f"SELECT count(*) FROM (SELECT {key_sql}, count(*) n FROM raw.{qi(table)} GROUP BY {key_sql} HAVING n>1)").fetchone()[0],
    })
    year_col = YEAR_COLUMNS.get(table)
    if year_col:
        years = set(con.execute(f"SELECT DISTINCT try_cast({qi(year_col)} AS INTEGER) FROM raw.{qi(table)} WHERE try_cast({qi(year_col)} AS INTEGER) IS NOT NULL").df().iloc[:,0])
        coverage.append({"table":table, "min_year":min(years), "max_year":max(years), "missing_years":sorted(EXPECTED_YEARS-years), "outside_scope":sorted(years-EXPECTED_YEARS)})
    # Identifier fields plus a bounded sample of wide value fields keeps the audit runnable.
    profile_cols = list(dict.fromkeys([*key, *(["NAME"] if "NAME" in cols else []), *cols[:40]]))
    for col in profile_cols:
        m = con.execute(f"SELECT count(*) FROM raw.{qi(table)} WHERE {qi(col)} IS NULL OR trim(cast({qi(col)} AS VARCHAR))=''").fetchone()[0]
        if m: missing.append({"table":table,"column":col,"missing_rows":m,"missing_pct":round(100*m/n,2)})
display(pd.DataFrame(inventory))
display(pd.DataFrame(coverage))
display(pd.DataFrame(key_checks))
display(pd.DataFrame(missing).sort_values("missing_pct",ascending=False).head(100) if missing else pd.DataFrame())


,table,rows,columns
0,climate,1154160,13


,table,min_year,max_year,missing_years,outside_scope
0,climate,2010,2024,[],[]


,table,candidate_key,invalid_key_rows,duplicate_key_groups
0,climate,fips + parameter + year_month,0,0


,table,column,missing_rows,missing_pct
0,climate,anomaly,1154160,100.0
1,climate,rank,1154160,100.0


## Source-specific checks

In [3]:
display(con.execute("""
SELECT parameter,parameter_label,unit,count(*) AS row_count,count(DISTINCT fips) AS counties,
 min(try_cast(year AS INTEGER)) AS min_year,max(try_cast(year AS INTEGER)) AS max_year,
 count(*) FILTER (WHERE try_cast(value AS DOUBLE) IS NULL) AS nonnumeric_values,
 min(try_cast(value AS DOUBLE)) AS minimum,max(try_cast(value AS DOUBLE)) AS maximum
FROM raw.climate GROUP BY 1,2,3 ORDER BY 1
""").df())
display(con.execute("""
SELECT fips,year,count(DISTINCT year_month||':'||parameter) AS observations
FROM raw.climate WHERE try_cast(year AS INTEGER) BETWEEN 2010 AND 2024
GROUP BY 1,2 HAVING observations<>48 ORDER BY observations,fips,year LIMIT 100
""").df())

,parameter,parameter_label,unit,row_count,counties,min_year,max_year,nonnumeric_values,minimum,maximum
0,pcp,precipitation,inch,288540,1603,2010,2024,0,0.0,40.45
1,tavg,average_temperature,deg_f,288540,1603,2010,2024,0,-7.9,98.10
2,tmax,maximum_temperature,deg_f,288540,1603,2010,2024,0,1.4,112.20
3,tmin,minimum_temperature,deg_f,288540,1603,2010,2024,0,-17.1,84.00


,fips,year,observations


## Interpretation and limitations

County monthly aggregates may mask within-county variation. Missing anomaly/rank fields can be source-expected. Completeness must be evaluated by parameter and month.

Displayed failures are investigation items. This notebook does not alter source data.

In [4]:
con.close()